# 02 Feature Engineering & Boruta Selection
Generating technical indicators, lags, and performing feature selection.

In [ ]:
import pandas as pd
from src.feature_engineering import create_feature_matrix, normalize_features
from src.boruta_selection import run_boruta_selection, plot_boruta_results

adj_close = pd.read_csv('../data/processed/adj_close.csv', index_col=0, parse_dates=True)
log_returns = pd.read_csv('../data/processed/log_returns.csv', index_col=0, parse_dates=True)

feat_matrix = create_feature_matrix(log_returns, adj_close)
X = feat_matrix.drop(columns=['^STOXX50E_return'])
y = (feat_matrix['^STOXX50E_return'] < feat_matrix['^STOXX50E_return'].quantile(0.05)).astype(int)

print("Running Boruta Selection (this may take a while)...")
selected, tentative, selector = run_boruta_selection(X, y)
print(f"Selected Features: {selected}")

plot_boruta_results(X, selector, '../results/plots/boruta_ranking.png')

X_selected = X[selected]
X_scaled, scaler = normalize_features(X_selected, exclude_cols=[])
X_scaled.to_csv('../data/processed/selected_features.csv')
pd.Series(selected).to_csv('../data/processed/selected_feature_names.csv', index=False)